In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import glob
import os
from pathlib import Path

import duckdb as dd
import pandas as pd

from src.config import *
from src.data.dqa import data_quality_assessment
from src.data.schema import CreditCardBalanceSchema

In [13]:
# Establish DuckDB connection
os.chdir(DATABASE_DIR)
con = dd.connect(HOME_CREDIT_DB)
con

# **Data Sourcing**

<https://www.kaggle.com/competitions/home-credit-default-risk/data>

In [4]:
# Read parquet files in data directory
os.chdir(HOME_CREDIT_DATA_DIR)
files_no_ext = [file.replace(".parquet", "") for file in glob.glob("*.parquet")]

print(files_no_ext)

['application_test', 'application_train', 'bureau', 'bureau_balance', 'credit_card_balance', 'installments_payments', 'pos_cash_balance', 'previous_application']


In [5]:
FILE_PATH = Path(HOME_CREDIT_DATA_DIR).as_posix()
print(FILE_PATH)

# Create tables in DuckDB from parquet files
for INPUT_FILE in files_no_ext:
    query = f"""
        CREATE OR REPLACE TABLE {INPUT_FILE} AS
        SELECT * FROM "{FILE_PATH}/{INPUT_FILE}.parquet"
    """

    con.execute(query)

C:/Users/mario/Documents/GitHub Repos/guides/data/raw/kaggle/home_credit_group


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
print(con.sql("SHOW ALL TABLES"))

┌─────────────────┬─────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

# **Data Dictionary**

In [7]:
os.chdir(HOME_CREDIT_DATA_DIR)
data_dict_df = pd.read_csv(DATA_DICTIONARY_FILE)

data_dict_df.columns = [
    col.strip().lower().replace(" ", "_") for col in data_dict_df.columns
]
data_dict_df.set_index("index", inplace=True)

data_dict_df.head()

,table,row,description,special
index,,,,
1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN


In [8]:
query = f"""
        CREATE OR REPLACE TABLE data_dictionary AS
        SELECT * FROM data_dict_df
    """

con.execute(query)

# **Data Quality Assessment (DQA)**

Following BCBS 239 principles, we implement a comprehensive Data Quality Assessment using Pandera to validate data across multiple dimensions:

| **Dimension** | **Implementation** | **Pandera Features Used** |
|---------------|-------------------|--------------------------|
| **Completeness** | Missing value analysis, nullable field validation | `nullable=True/False`, completeness statistics |
| **Accuracy** | Data type validation, range checks, format validation | `pa.Check.ge()`, `pa.Check.in_range()`, custom checks |
| **Uniqueness** | Primary key duplicate detection | `unique=True` field constraints |
| **Timeliness** | Date range validation, data freshness checks | Date column analysis |

In [9]:
# Re-establish connection for DQA
os.chdir(DATABASE_DIR)
con = dd.connect(HOME_CREDIT_DB)

# Get list of available tables
available_tables = con.execute("SHOW ALL TABLES").df()["name"].tolist()
print("Available tables for DQA:")
for i, table in enumerate(available_tables, 1):
    print(f"  {i}. {table}")

print(f"\nTotal tables: {len(available_tables)}")

Available tables for DQA:
  1. application_test
  2. application_train
  3. bureau
  4. bureau_balance
  5. credit_card_balance
  6. data_dictionary
  7. installments_payments
  8. pos_cash_balance
  9. previous_application

Total tables: 9


In [ ]:
# Perform DQA on credit card balance data
query = f"SELECT COUNT(*) as total_rows FROM credit_card_balance"
total_rows = con.execute(query).fetchone()[0]

if "credit_card_balance" in available_tables:
    success, validated_data = data_quality_assessment(
        "credit_card_balance", CreditCardBalanceSchema, con, sample_size=total_rows
    )

    if success:
        print("\n✅ Credit card balance data quality assessment completed successfully")
    else:
        print("\n❌ Credit card balance data quality assessment failed")
else:
    print("⚠️ credit_card_balance table not found")


DATA QUALITY ASSESSMENT: CREDIT_CARD_BALANCE

🔍 1. COMPLETENESS ASSESSMENT
----------------------------------------
Total records in credit_card_balance: 3,840,312
Sample size for validation: 3,840,312 rows

Completeness by column (% non-null):
  ✅ SK_ID_PREV: 100.0%
  ✅ SK_ID_CURR: 100.0%
  ✅ MONTHS_BALANCE: 100.0%
  ✅ AMT_BALANCE: 100.0%
  ✅ AMT_CREDIT_LIMIT_ACTUAL: 100.0%
  ⚠️ AMT_DRAWINGS_ATM_CURRENT: 80.5%
  ✅ AMT_DRAWINGS_CURRENT: 100.0%
  ⚠️ AMT_DRAWINGS_OTHER_CURRENT: 80.5%
  ⚠️ AMT_DRAWINGS_POS_CURRENT: 80.5%
  ⚠️ AMT_INST_MIN_REGULARITY: 92.1%
  ⚠️ AMT_PAYMENT_CURRENT: 80.0%
  ✅ AMT_PAYMENT_TOTAL_CURRENT: 100.0%
  ✅ AMT_RECEIVABLE_PRINCIPAL: 100.0%
  ✅ AMT_RECIVABLE: 100.0%
  ✅ AMT_TOTAL_RECEIVABLE: 100.0%
  ⚠️ CNT_DRAWINGS_ATM_CURRENT: 80.5%
  ✅ CNT_DRAWINGS_CURRENT: 100.0%
  ⚠️ CNT_DRAWINGS_OTHER_CURRENT: 80.5%
  ⚠️ CNT_DRAWINGS_POS_CURRENT: 80.5%
  ⚠️ CNT_INSTALMENT_MATURE_CUM: 92.1%
  ✅ NAME_CONTRACT_STATUS: 100.0%
  ✅ SK_DPD: 100.0%
  ✅ SK_DPD_DEF: 100.0%

🔍 2. SCHEMA V

In [21]:
con.close()